# Traffic Density Estimation using CSRNet on TRANCOS Dataset

This notebook guides you through setting up, training, and running inference on the CSRNet traffic density estimation model in Google Colab using GPU acceleration.

### Steps:
1. Verify GPU availability.
2. Mount Google Drive to persist checkpoints.
3. Set up workspace paths.
4. Train the model on GPU.
5. Evaluate the model on the TRANCOS test set.
6. Run inference on a custom image with inline visualization.

In [ ]:
# Step 1: Verify GPU Availability
import torch
gpu_available = torch.cuda.is_available()
print("GPU Available:", gpu_available)
if gpu_available:
    print("Device Name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: GPU is not enabled. Go to Edit -> Notebook settings -> Hardware accelerator and select GPU.")

## Step 2: Mount Google Drive
Mounting Google Drive allows you to persistent-save checkpoints (`best_model.pth` and `last_model.pth`) and load your dataset directly from Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 3: Configure Workspace Paths

Ensure you have uploaded the project files and dataset directory to your Google Drive.
Specify the workspace path to your uploaded project directory (e.g. `/content/drive/MyDrive/Traffic_Density_Estimation`).

In [ ]:
import os
import sys

# Update this path to the location where you uploaded the files on Google Drive
WORKSPACE_PATH = "/content/drive/MyDrive/Traffic_Monitoring/density_estimation"

# Change directory to the workspace
%cd {WORKSPACE_PATH}

# Add the workspace path to sys.path so modules can import correctly
if WORKSPACE_PATH not in sys.path:
    sys.path.append(WORKSPACE_PATH)

# Verify files inside the workspace
!ls -la TrafficDensity

## Step 4: Train CSRNet

We will run the training script. We will utilize VGG-16 pretrained weights, and save checkpoints directly to Google Drive. This ensures that even if Colab disconnects, your training checkpoint will not be lost and you can resume training later using the `--resume` flag.

In [ ]:
# Run training on GPU
# Adjust epochs, batch_size, and learning rate as needed
!python TrafficDensity/train.py \
    --dataset_path "TRANCOS - edited/train_data" \
    --checkpoint_path "TrafficDensity/checkpoints" \
    --epochs 100 \
    --batch_size 8 \
    --lr 1e-5 \
    --sigma 4.0

### Optional: Resume Training from Checkpoint
If your runtime disconnected or you want to train for more epochs, you can resume training by pointing to the `last_model.pth` checkpoint file:

In [ ]:
# Resume training
# !python TrafficDensity/train.py \
#     --dataset_path "TRANCOS - edited/train_data" \
#     --checkpoint_path "TrafficDensity/checkpoints" \
#     --resume "TrafficDensity/checkpoints/last_model.pth" \
#     --epochs 100 \
#     --batch_size 8 \
#     --lr 1e-5

## Step 5: Evaluate Model on the Test Dataset
Run evaluation on the test split split of TRANCOS. This will load the `best_model.pth` checkpoint and report MAE, RMSE, and bias count errors.

In [ ]:
!python TrafficDensity/evaluate.py \
    --dataset_path "TRANCOS - edited/test_data" \
    --model_path "TrafficDensity/checkpoints/best_model.pth"

## Step 6: Run Inference on a Single Image and Visualize
Execute prediction on any query image and render the side-by-side 4-panel visual result (original image, greyscale density map, jet heatmap, and translucent overlay) inline inside the notebook.

In [ ]:
import cv2
import matplotlib.pyplot as plt

# Select a test image to predict on
IMAGE_PATH = "TRANCOS - edited/test_data/images/image-1-000333.jpg"
SAVE_PATH = "TrafficDensity/prediction_result.png"

# Run prediction script headlessly (saving visualization to prediction_result.png)
!python TrafficDensity/predict.py \
    --image_path {IMAGE_PATH} \
    --model_path "TrafficDensity/checkpoints/best_model.pth" \
    --save_path {SAVE_PATH}

# Load and display the generated visualization plot inline
prediction_img = cv2.imread(SAVE_PATH)
if prediction_img is not None:
    prediction_img = cv2.cvtColor(prediction_img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(20, 6))
    plt.imshow(prediction_img)
    plt.axis("off")
    plt.show()
else:
    print("Error loading prediction plot image. Make sure predict.py ran successfully.")